In [1]:
# Imports and Constants 

import requests
import pandas as pd
import time
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()
CENSUS_API_KEY = os.getenv("CENSUS_API_KEY")

# Constants 
TEHAMA_COUNTY_FIPS = "103"
STATE_FIPS         = "06"

# CBP available years (2023 and 2024 not yet released)
CBP_YEARS = list(range(2012, 2024))  # 2012-2023

# Output folder
RAW_DIR = Path("data/raw/cbp")
RAW_DIR.mkdir(parents=True, exist_ok=True)

print(" Setup complete")
print(f"   Years  : {CBP_YEARS[0]} → {CBP_YEARS[-1]}")
print(f"   Total  : {len(CBP_YEARS)} years")
print(f"   Output : {RAW_DIR}")

 Setup complete
   Years  : 2012 → 2023
   Total  : 12 years
   Output : data\raw\cbp


In [2]:
# CBP Fetch Function 

def fetch_cbp(year, geography):
    """
    Fetch County Business Patterns data.
    
    Returns total employment and establishments
    across all industries for given geography and year.
    """

    # Build geography parameter 
    if geography == "county":
        geo_param = (f"for=county:{TEHAMA_COUNTY_FIPS}"
                     f"&in=state:{STATE_FIPS}")
        label = "Tehama County"
        fips  = "06103"
    elif geography == "state":
        geo_param = f"for=state:{STATE_FIPS}"
        label = "California"
        fips  = STATE_FIPS
    elif geography == "nation":
        geo_param = "for=us:1"
        label = "United States"
        fips  = "US"

    # CBP uses different NAICS versions in different years
    # 2012-2016: NAICS2012
    # 2017+:     NAICS2017
    if year >= 2017:
        naics_param = "NAICS2017=00"
        naics_var   = "NAICS2017"
    elif year >= 2012:
        naics_param = "NAICS2012=00"
        naics_var   = "NAICS2012"
    else:
        naics_param = "NAICS2007=00"
        naics_var   = "NAICS2007"

    url = (f"https://api.census.gov/data/{year}/cbp"
           f"?get=NAME,EMP,ESTAB,{naics_var}"
           f"&{geo_param}"
           f"&{naics_param}"
           f"&key={CENSUS_API_KEY}")

    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    row = dict(zip(data[0], data[1]))

    result = {
        "geography"      : label,
        "geo_type"       : geography,
        "cbp_year"       : year,
        "fips"           : fips,
        "total_employment"      : pd.to_numeric(row.get("EMP"),   errors="coerce"),
        "employer_establishments": pd.to_numeric(row.get("ESTAB"), errors="coerce"),
        "fetched_at"     : pd.Timestamp.now('UTC').isoformat(),
    }

    return pd.DataFrame([result])

In [3]:
# Fetch All CBP Data (Tehama Only) 

import time

frames = []
errors = []

print("FETCHING CBP DATA FOR TEHAMA COUNTY")
print("=" * 50)
print(f"Years      : {CBP_YEARS[0]} → {CBP_YEARS[-1]}")
print(f"Total calls: {len(CBP_YEARS)}")
print("=" * 50)

for year in CBP_YEARS:
    try:
        df = fetch_cbp(year, "county")
        frames.append(df)
        print(f"{year}")
        time.sleep(0.5)
    except Exception as e:
        print(f"{year} failed: {e}")
        errors.append(f"{year}: {e}")

# Combine 
cbp_df = pd.concat(frames, ignore_index=True)

print("\n" + "=" * 50)
print(f"Fetch complete")
print(f"   Total rows : {len(cbp_df)}")
print(f"   Years      : {cbp_df['cbp_year'].min()} → "
      f"{cbp_df['cbp_year'].max()}")

if errors:
    print(f"\n{len(errors)} errors:")
    for e in errors:
        print(f"   {e}")

FETCHING CBP DATA FOR TEHAMA COUNTY
Years      : 2012 → 2023
Total calls: 12
2012
2013
2014
2015
2016
2017
2018
2019
2020
2021
2022
2023

Fetch complete
   Total rows : 12
   Years      : 2012 → 2023


In [4]:
# Save CBP Data 

parquet_path = RAW_DIR / "cbp_raw.parquet"
csv_path     = RAW_DIR / "cbp_raw.csv"

cbp_df.to_parquet(parquet_path, index=False)
cbp_df.to_csv(csv_path, index=False)

print(f"Parquet saved : {parquet_path}")
print(f"\n CSV saved     : {csv_path}")

Parquet saved : data\raw\cbp\cbp_raw.parquet

 CSV saved     : data\raw\cbp\cbp_raw.csv


In [5]:
df=pd.read_parquet("Data/raw/cbp/cbp_raw.parquet")
df

,geography,geo_type,cbp_year,fips,total_employment,employer_establishments,fetched_at
0,Tehama County,county,2012,06103,11095,975,2026-04-22T23:44:07.706945+00:00
1,Tehama County,county,2013,06103,11381,956,2026-04-22T23:44:09.368622+00:00
2,Tehama County,county,2014,06103,11723,958,2026-04-22T23:44:11.030342+00:00
3,Tehama County,county,2015,06103,12306,981,2026-04-22T23:44:12.674128+00:00
4,Tehama County,county,2016,06103,12628,975,2026-04-22T23:44:14.289622+00:00
5,Tehama County,county,2017,06103,12305,976,2026-04-22T23:44:15.887864+00:00
6,Tehama County,county,2018,06103,12513,967,2026-04-22T23:44:20.655965+00:00
7,Tehama County,county,2019,06103,13098,994,2026-04-22T23:44:22.248217+00:00
8,Tehama County,county,2020,06103,13429,982,2026-04-22T23:44:24.040551+00:00
9,Tehama County,county,2021,06103,13123,1009,2026-04-22T23:44:25.774546+00:00
